<a href="https://www.kaggle.com/code/rishabhbhartiya/s6e11-credit-card-fraud-target-encoding-glm-b?scriptVersionId=375174192" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Playground S6E11 - Credit Card Fraud: target-encoding + GLM-boost (v2)
Metric: ROC AUC. Folds: `StratifiedKFold(5, shuffle=True, random_state=42)` (same split as the public OOF banks).

**Why this version.** The label is almost additive, and the generator leaves fingerprints in the float columns
(repeated exact values, first/second decimal digits correlated with the label). Raw-feature trees barely see them.
So every key is target-encoded **strictly in-fold**, then fed to:
1. a logistic regression (GLM) on the encoding logits,
2. a 4-leaf LightGBM boosted from the GLM margin (`init_score`),
3. a 3-leaf LightGBM on encodings + raw features.

Test predictions come from a full-data refit (fold-mean predictions are kept as a sanity check).
Recipe credits (from the competition discussion): goodpjw2008, alexchilton, megayak, evgendvorkin, DavutKursun.

In [1]:
# %% 0. Config
import os
SEED, NFOLD = 42, 5
M_SMOOTH   = 30            # m-estimate smoothing for target encoding
REFIT      = True          # final test predictions from a full-data refit
LEAK_CHECK = True          # label-shuffle test through the encoder + GLM (AUC should be ~0.5)
DIR = "/kaggle/input/competitions/playground-series-s6e11"
OUT = "/kaggle/working"

In [2]:
# %% 1. Imports + data
import warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

TARGET, ID = "is_fraud", "id"
train = pd.read_csv(f"{DIR}/train.csv")
test  = pd.read_csv(f"{DIR}/test.csv")
sub   = pd.read_csv(f"{DIR}/sample_submission.csv")
y = train[TARGET].values.astype(np.float64)
n_tr, n_te = len(train), len(test)
ALL = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
y_all = np.r_[y, np.full(n_te, np.nan)]          # test labels unknown
test_idx = np.arange(n_tr, n_tr + n_te)
print(train.shape, test.shape, "fraud rate:", y.mean())

(699960, 26) (299984, 25) fraud rate: 0.016699525687182123


In [3]:
# %% 2. Keys to target-encode + plain numeric features
FLOAT_COLS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km",
              "account_balance_usd", "velocity_score", "merchant_risk_score"]
INT_COLS   = ["txn_count_last_24h", "card_age_months", "customer_age", "cvv_retry_count",
              "time_of_day_hour", "day_of_week", "prior_disputes"]
BOOL_COLS  = ALL.select_dtypes("bool").columns.tolist()
CAT_COLS   = ALL.select_dtypes("object").columns.tolist()

key_codes = {}                                    # key name -> integer code per row (train+test)
def add_key(name, s):
    codes, uniq = pd.factorize(s)
    if len(uniq) >= 2:                            # constant keys carry nothing
        key_codes[name] = codes.astype(np.int64)

for c in CAT_COLS:  add_key(f"cat_{c}",  ALL[c])
for c in BOOL_COLS: add_key(f"bool_{c}", ALL[c].astype(int))
for c in INT_COLS:  add_key(f"int_{c}",  ALL[c])

cents = {}
for c in FLOAT_COLS:
    cents[c] = np.rint(ALL[c].values * 100).astype(np.int64)
    add_key(f"exact_{c}", cents[c])                       # every exact value
    add_key(f"d1_{c}",    (cents[c] // 10) % 10)          # first decimal digit
    add_key(f"d2_{c}",    cents[c] % 10)                  # second decimal digit

KEYS  = list(key_codes)
K     = len(KEYS)
key_n = {k: int(key_codes[k].max()) + 1 for k in KEYS}

# plain numeric block for the trees (raw values, cents as numbers, exact-value counts, flag sum)
num = {}
for c in INT_COLS + FLOAT_COLS: num[c] = ALL[c].values
for c in BOOL_COLS:             num[c] = ALL[c].astype(int).values
for c in FLOAT_COLS:
    num[f"cents_{c}"] = (cents[c] % 100).astype(float)
    num[f"cnt_{c}"]   = np.bincount(cents[c])[cents[c]].astype(float)   # unsupervised count over train+test
num["risk_flags"] = ALL[BOOL_COLS].astype(int).sum(axis=1).values
NUM = pd.DataFrame(num).values.astype(np.float32)
print("target-encoded keys:", K, "| plain numeric features:", NUM.shape[1])

target-encoded keys: 36 | plain numeric features: 32


In [4]:
# %% 3. In-fold target encoding (m-estimate) as logits
def te_fit_apply(fit_idx, apply_idx):
    """Encode rows apply_idx using label statistics of fit_idx only."""
    yf = y_all[fit_idx]
    prior = yf.mean()
    out = np.empty((len(apply_idx), K))
    for j, k in enumerate(KEYS):
        c = key_codes[k]
        cnt = np.bincount(c[fit_idx], minlength=key_n[k])
        pos = np.bincount(c[fit_idx], weights=yf, minlength=key_n[k])
        enc = (pos + M_SMOOTH * prior) / (cnt + M_SMOOTH)
        out[:, j] = enc[c[apply_idx]]
    out = np.clip(out, 1e-6, 1 - 1e-6)
    return np.log(out / (1 - out))

def te_oof(tr_idx, seed):
    """Training rows are encoded out-of-fold INSIDE the training part (no row sees its own label)."""
    out = np.empty((len(tr_idx), K))
    for a, b in StratifiedKFold(NFOLD, shuffle=True, random_state=seed).split(tr_idx, y_all[tr_idx]):
        out[b] = te_fit_apply(tr_idx[a], tr_idx[b])
    return out

def fit_glm(Ttr, ytr):
    sc = StandardScaler().fit(Ttr)
    glm = LogisticRegression(C=1.0, max_iter=300).fit(sc.transform(Ttr), ytr)
    return lambda T: glm.decision_function(sc.transform(T))      # returns margin (logit)

def feats(T, idx):
    return np.hstack([T, NUM[idx]]).astype(np.float32)

In [5]:
# %% 4. Optional leak check: shuffle labels, run the whole encoder + GLM, expect AUC ~ 0.5
if LEAK_CHECK:
    y_real = y_all.copy()
    y_all[:n_tr] = np.random.default_rng(0).permutation(y)
    tr0, va0 = next(StratifiedKFold(NFOLD, shuffle=True, random_state=SEED).split(np.zeros(n_tr), y_all[:n_tr]))
    g = fit_glm(te_oof(tr0, SEED + 1), y_all[tr0])
    print("label-shuffle AUC (should be ~0.50):", roc_auc_score(y_all[va0], g(te_fit_apply(tr0, va0))))
    y_all[:] = y_real

label-shuffle AUC (should be ~0.50): 0.501458833965157


In [6]:
# %% 5. Models: GLM, GLM-boost (4 leaves), LGB 3 leaves  -- 5-fold OOF
P_BASE = dict(objective="binary", metric="auc", learning_rate=0.05, min_data_in_leaf=100,
              feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
              verbose=-1, seed=SEED)
P_BOOST = dict(P_BASE, num_leaves=4)
P_LGB3  = dict(P_BASE, num_leaves=3)

def train_lgb(Xtr, ytr, init_tr, params, Xev=None, yev=None, init_ev=None, rounds=None):
    dtr = lgb.Dataset(Xtr, ytr, init_score=init_tr)
    if rounds is None:
        dev = lgb.Dataset(Xev, yev, init_score=init_ev, reference=dtr)
        return lgb.train(params, dtr, 5000, valid_sets=[dev],
                         callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train(params, dtr, rounds)

names = ["glm", "glm_boost", "lgb3"]
oof   = {m: np.zeros(n_tr) for m in names}
tfold = {m: np.zeros(n_te) for m in names}
best_it = {"glm_boost": [], "lgb3": []}

skf = StratifiedKFold(NFOLD, shuffle=True, random_state=SEED)
for f, (tr, va) in enumerate(skf.split(np.zeros(n_tr), y)):
    Ttr, Tva, Tte = te_oof(tr, SEED + 1), te_fit_apply(tr, va), te_fit_apply(tr, test_idx)
    Xtr, Xva, Xte = feats(Ttr, tr), feats(Tva, va), feats(Tte, test_idx)

    glm = fit_glm(Ttr, y[tr])
    m_tr, m_va, m_te = glm(Ttr), glm(Tva), glm(Tte)
    oof["glm"][va] = m_va;  tfold["glm"] += m_te / NFOLD

    b = train_lgb(Xtr, y[tr], m_tr, P_BOOST, Xva, y[va], m_va)
    oof["glm_boost"][va] = b.predict(Xva, raw_score=True) + m_va
    tfold["glm_boost"]  += (b.predict(Xte, raw_score=True) + m_te) / NFOLD
    best_it["glm_boost"].append(b.best_iteration)

    b = train_lgb(Xtr, y[tr], None, P_LGB3, Xva, y[va], None)
    oof["lgb3"][va] = b.predict(Xva, raw_score=True)
    tfold["lgb3"] += b.predict(Xte, raw_score=True) / NFOLD
    best_it["lgb3"].append(b.best_iteration)

    print(f"fold {f}: " + " | ".join(f"{m} {roc_auc_score(y[va], oof[m][va]):.5f}" for m in names))

for m in names:
    print(f"[{m}] OOF AUC = {roc_auc_score(y, oof[m]):.5f}")
print("best iterations:", best_it)

fold 0: glm 0.94325 | glm_boost 0.94735 | lgb3 0.94726
fold 1: glm 0.94597 | glm_boost 0.95003 | lgb3 0.94997
fold 2: glm 0.95147 | glm_boost 0.95411 | lgb3 0.95430
fold 3: glm 0.94628 | glm_boost 0.94904 | lgb3 0.94896
fold 4: glm 0.93889 | glm_boost 0.94350 | lgb3 0.94354
[glm] OOF AUC = 0.94513
[glm_boost] OOF AUC = 0.94878
[lgb3] OOF AUC = 0.94875
best iterations: {'glm_boost': [829, 886, 961, 783, 915], 'lgb3': [1515, 1393, 1223, 968, 1228]}


In [7]:
# %% 6. Full-data refit for the test predictions (rounds = 1.1 x mean best iteration)
if REFIT:
    all_tr = np.arange(n_tr)
    Ttr, Tte = te_oof(all_tr, SEED + 2), te_fit_apply(all_tr, test_idx)
    Xtr, Xte = feats(Ttr, all_tr), feats(Tte, test_idx)
    glm = fit_glm(Ttr, y)
    m_tr, m_te = glm(Ttr), glm(Tte)
    test_pred = {"glm": m_te}
    b = train_lgb(Xtr, y, m_tr, P_BOOST, rounds=int(1.1 * np.mean(best_it["glm_boost"])))
    test_pred["glm_boost"] = b.predict(Xte, raw_score=True) + m_te
    b = train_lgb(Xtr, y, None, P_LGB3, rounds=int(1.1 * np.mean(best_it["lgb3"])))
    test_pred["lgb3"] = b.predict(Xte, raw_score=True)
    # sanity: refit and fold-mean predictions must agree closely in rank
    for m in names:
        rho = np.corrcoef(rankdata(test_pred[m]), rankdata(tfold[m]))[0, 1]
        print(f"[{m}] rank corr refit vs fold-mean: {rho:.5f}")
        assert rho > 0.98, f"refit looks broken for {m}"
else:
    test_pred = tfold

[glm] rank corr refit vs fold-mean: 0.99949
[glm_boost] rank corr refit vs fold-mean: 0.99749
[lgb3] rank corr refit vs fold-mean: 0.99716


In [8]:
# %% 7. Rank blend -> submission.csv (+ save OOF/test for later stacking)
oofs  = np.column_stack([rankdata(oof[m]) / n_tr for m in names])
tests = np.column_stack([rankdata(test_pred[m]) / n_te for m in names])
neg = lambda w: -roc_auc_score(y, oofs @ (np.abs(w) / np.abs(w).sum()))
r = minimize(neg, np.ones(len(names)) / len(names), method="Nelder-Mead")
w = np.abs(r.x) / np.abs(r.x).sum()
print({m: round(float(x), 3) for m, x in zip(names, w)}, "| blend OOF AUC:", round(-r.fun, 5))

sub[TARGET] = tests @ w
sub.to_csv(f"{OUT}/submission.csv", index=False)
pd.DataFrame({ID: train[ID], **{m: oof[m] for m in names}}).to_csv(f"{OUT}/oof_own.csv", index=False)
pd.DataFrame({ID: test[ID],  **{m: test_pred[m] for m in names}}).to_csv(f"{OUT}/test_own.csv", index=False)

chk = pd.read_csv(f"{OUT}/submission.csv")
assert list(chk.columns) == [ID, TARGET] and len(chk) == n_te and chk[ID].equals(test[ID])
assert chk[TARGET].notna().all() and chk[TARGET].between(0, 1).all()
print(chk.head(), "\nfiles in output dir:", os.listdir(OUT))

{'glm': 0.0, 'glm_boost': 0.51, 'lgb3': 0.49} | blend OOF AUC: 0.94903
       id  is_fraud
0  699960  0.149134
1  699961  0.055536
2  699962  0.957580
3  699963  0.558250
4  699964  0.446549 
files in output dir: ['__notebook__.ipynb', 'submission.csv', 'test_own.csv', 'oof_own.csv']
